In [8]:
import dspy
import pandas as pd
import numpy as np
# import duckdb

import os
from dotenv import load_dotenv
load_dotenv()


C:\Users\arsla\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [9]:
# from .autonotebook import tqdm as notebook_tqdm
# !pip install duckdb




In [10]:
class create_schema(dspy.Signature):
    """
    You are a schema generation assistant. Given a natural language description of the data or entities
    the user wants to store, generate a SQL CREATE TABLE schema that defines appropriate tables, columns, 
    and data types. Use sensible names, appropriate data types, and include primary keys. If multiple tables 
    are needed, include foreign key relationships where applicable. Return only the SQL schema.

    Example:
    User Prompt: "I want to store information about books, authors, and publishers. Each book has a title, 
    publication year, genre, and is written by one or more authors. Each author has a name and birth year. 
    Each publisher has a name and address."

    Output: A valid SQL schema containing CREATE TABLE statements to represent this data model.

    Your are using duckDB SQL, which is based on SQLite
    - DO NOT TRY to add foreign_key etc relationships
    
    
    """
    user_prompt = dspy.InputField(desc="The prompt the user has given on what schema they want you to generate")
    schema_sql = dspy.OutputField(desc="The SCHEMA SQL for the requested prompt")

In [11]:
# Load a language model for dspy
# You can use dspy.OpenAI, dspy.HuggingFace, or another supported backend.
# Example with OpenAI (requires OPENAI_API_KEY env var):
lm = dspy.LM(model="openai/gpt-4o-mini", max_tokens=1024)
dspy.settings.configure(lm=lm)


In [12]:
schema_gen = dspy.Predict(create_schema)
response=schema_gen(user_prompt= "Giving me the schema of a ecommerce website")

print(response.schema_sql)

23:37:22 - LiteLLM:ERROR: caching.py:563 - LiteLLM Cache: Excepton add_cache: __annotations__
Traceback (most recent call last):
  File "C:\Users\arsla\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\litellm\caching\caching.py", line 558, in add_cache
    cache_key, cached_data, kwargs = self._add_cache_logic(
                                     ~~~~~~~~~~~~~~~~~~~~~^
        result=result, **kwargs
        ^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "C:\Users\arsla\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\litellm\caching\caching.py", line 542, in _add_cache_logic
    raise e
  File "C:\Users\arsla\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\litellm\caching\caching.py", line 522, in _add_cache_logic
    cache_key = self.get_cache_key(**kwargs)
  File 

CREATE TABLE Users (
    user_id INTEGER PRIMARY KEY,
    username TEXT NOT NULL,
    password TEXT NOT NULL,
    email TEXT NOT NULL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE Products (
    product_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    description TEXT,
    price DECIMAL(10, 2) NOT NULL,
    stock_quantity INTEGER NOT NULL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE Orders (
    order_id INTEGER PRIMARY KEY,
    user_id INTEGER NOT NULL,
    order_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    total_amount DECIMAL(10, 2) NOT NULL,
    FOREIGN KEY (user_id) REFERENCES Users(user_id)
);

CREATE TABLE Order_Items (
    order_item_id INTEGER PRIMARY KEY,
    order_id INTEGER NOT NULL,
    product_id INTEGER NOT NULL,
    quantity INTEGER NOT NULL,
    price DECIMAL(10, 2) NOT NULL,
    FOREIGN KEY (order_id) REFERENCES Orders(order_id),
    FOREIGN KEY (product_id) REFERENCES Products(product_id)
);

CREATE TABLE Categories 

23:37:22 - LiteLLM:ERROR: litellm_logging.py:3482 - Error creating standard logging object - __annotations__
Traceback (most recent call last):
  File "C:\Users\arsla\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\litellm\litellm_core_utils\litellm_logging.py", line 3464, in get_standard_logging_object_payload
    model_parameters=ModelParamHelper.get_standard_logging_model_parameters(
                     ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        kwargs.get("optional_params", None) or {}
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ),
    ^
  File "C:\Users\arsla\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\litellm\litellm_core_utils\model_param_helper.py", line 28, in get_standard_logging_model_parameters
    ModelParamHelper._get_relevant_args_to_use_for_logging()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

In [13]:
import duckdb
conn = duckdb.connect(database=':memory:')




In [14]:
print(response.schema_sql.replace('```','').replace('sql',''))

CREATE TABLE Users (
    user_id INTEGER PRIMARY KEY,
    username TEXT NOT NULL,
    password TEXT NOT NULL,
    email TEXT NOT NULL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE Products (
    product_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    description TEXT,
    price DECIMAL(10, 2) NOT NULL,
    stock_quantity INTEGER NOT NULL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE Orders (
    order_id INTEGER PRIMARY KEY,
    user_id INTEGER NOT NULL,
    order_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    total_amount DECIMAL(10, 2) NOT NULL,
    FOREIGN KEY (user_id) REFERENCES Users(user_id)
);

CREATE TABLE Order_Items (
    order_item_id INTEGER PRIMARY KEY,
    order_id INTEGER NOT NULL,
    product_id INTEGER NOT NULL,
    quantity INTEGER NOT NULL,
    price DECIMAL(10, 2) NOT NULL,
    FOREIGN KEY (order_id) REFERENCES Orders(order_id),
    FOREIGN KEY (product_id) REFERENCES Products(product_id)
);

CREATE TABLE Categories 

In [15]:
# Remove unsupported actions from FOREIGN KEY constraints before 
cleaned_sql = response.schema_sql.replace('```','').replace('sql','')
result = conn.execute(cleaned_sql)

In [16]:
print(conn.execute("SHOW TABLES").df())



                 name
0          Categories
1         Order_Items
2              Orders
3  Product_Categories
4            Products
5             Reviews
6               Users


In [17]:
class populate_table(dspy.Signature):
    """
    You are provided with a DuckDB SQL table schema.

    Your task is to write complete Python code that:
    - Uses DuckDB in Python.
    - Generates 250 rows of realistic simulated data based on column types and names.
    - Uses libraries such as `faker`, `random`, or `numpy` for data generation.
    - Creates the table using the exact schema provided.
    - Inserts the generated rows using DuckDB SQL INSERT statements (no DataFrame insertion).
    - Uses parameterized queries to avoid SQL injection and ensure clean formatting.
    - No need to import duckdb or connect it is already connected as conn
    - Do not do conn = duckdb.connect(), it is already connected
    - Take care of the foreign key relations, ensuring you add in good sequence!

    Do not return anything except the Python code.

    One-shot Example:

    Input
    table_schema = '''
    CREATE TABLE users (
        user_id INTEGER,
        full_name VARCHAR,
        email VARCHAR,
        age INTEGER,
        join_date DATE,
        is_active BOOLEAN
    );
    '''

    Output
    python_code = '''
    from faker import Faker
    import random
    from datetime import datetime, timedelta

    # Initialize
    fake = Faker()


    # Insert 250 rows
    insert_query = "INSERT INTO users VALUES (?, ?, ?, ?, ?, ?)"
    for i in range(1, 251):
        full_name = fake.name()
        email = fake.email()
        age = random.randint(18, 70)
        join_date = fake.date_between(start_date='-3y', end_date='today').isoformat()
        is_active = random.choice([True, False])
        conn.execute(insert_query, (i, full_name, email, age, join_date, is_active))
    '''
    """
    table_schema = dspy.InputField(desc="The DuckDB SQL schema for the table")
    python_code = dspy.OutputField(desc="Python code that generates simulated data & adds it via DuckDB SQL")


In [18]:
populate_agent = dspy.Predict(populate_table)

# Fetch the schema for table Datasets using duckdb SQL
# import duckdb

# conn = duckdb.connect()
# Get the DDL for all tables in the database and print them
tables = [row[0] for row in conn.execute("SHOW TABLES").fetchall()]
schema_result = []
for table in tables:
    ddl = conn.execute(f"DESCRIBE {table}").fetchall()
    schema_result.append((table,ddl))
    print(f"Schema for table '{table}':\n{ddl}\n{'-'*40}")



Schema for table 'Categories':
[('category_id', 'INTEGER', 'NO', 'PRI', None, None), ('category_name', 'VARCHAR', 'NO', None, None, None)]
----------------------------------------
Schema for table 'Order_Items':
[('order_item_id', 'INTEGER', 'NO', 'PRI', None, None), ('order_id', 'INTEGER', 'NO', None, None, None), ('product_id', 'INTEGER', 'NO', None, None, None), ('quantity', 'INTEGER', 'NO', None, None, None), ('price', 'DECIMAL(10,2)', 'NO', None, None, None)]
----------------------------------------
Schema for table 'Orders':
[('order_id', 'INTEGER', 'NO', 'PRI', None, None), ('user_id', 'INTEGER', 'NO', None, None, None), ('order_date', 'TIMESTAMP', 'YES', None, 'CURRENT_TIMESTAMP', None), ('total_amount', 'DECIMAL(10,2)', 'NO', None, None, None)]
----------------------------------------
Schema for table 'Product_Categories':
[('product_id', 'INTEGER', 'NO', 'PRI', None, None), ('category_id', 'INTEGER', 'NO', 'PRI', None, None)]
----------------------------------------
Schema fo

In [19]:
schema_result

[('Categories',
  [('category_id', 'INTEGER', 'NO', 'PRI', None, None),
   ('category_name', 'VARCHAR', 'NO', None, None, None)]),
 ('Order_Items',
  [('order_item_id', 'INTEGER', 'NO', 'PRI', None, None),
   ('order_id', 'INTEGER', 'NO', None, None, None),
   ('product_id', 'INTEGER', 'NO', None, None, None),
   ('quantity', 'INTEGER', 'NO', None, None, None),
   ('price', 'DECIMAL(10,2)', 'NO', None, None, None)]),
 ('Orders',
  [('order_id', 'INTEGER', 'NO', 'PRI', None, None),
   ('user_id', 'INTEGER', 'NO', None, None, None),
   ('order_date', 'TIMESTAMP', 'YES', None, 'CURRENT_TIMESTAMP', None),
   ('total_amount', 'DECIMAL(10,2)', 'NO', None, None, None)]),
 ('Product_Categories',
  [('product_id', 'INTEGER', 'NO', 'PRI', None, None),
   ('category_id', 'INTEGER', 'NO', 'PRI', None, None)]),
 ('Products',
  [('product_id', 'INTEGER', 'NO', 'PRI', None, None),
   ('name', 'VARCHAR', 'NO', None, None, None),
   ('description', 'VARCHAR', 'YES', None, None, None),
   ('price', 'DEC

In [20]:
response = populate_agent(table_schema = str(schema_result))

print(response.python_code)

23:37:35 - LiteLLM:ERROR: caching.py:563 - LiteLLM Cache: Excepton add_cache: __annotations__
Traceback (most recent call last):
  File "C:\Users\arsla\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\litellm\caching\caching.py", line 558, in add_cache
    cache_key, cached_data, kwargs = self._add_cache_logic(
                                     ~~~~~~~~~~~~~~~~~~~~~^
        result=result, **kwargs
        ^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "C:\Users\arsla\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\litellm\caching\caching.py", line 542, in _add_cache_logic
    raise e
  File "C:\Users\arsla\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\litellm\caching\caching.py", line 522, in _add_cache_logic
    cache_key = self.get_cache_key(**kwargs)
  File 

from faker import Faker
import random
from datetime import datetime, timedelta

# Initialize
fake = Faker()

# Insert Categories
insert_category_query = "INSERT INTO Categories VALUES (?, ?)"
for i in range(1, 11):
    category_name = fake.word().capitalize()
    conn.execute(insert_category_query, (i, category_name))

# Insert Users
insert_user_query = "INSERT INTO Users VALUES (?, ?, ?, ?, ?)"
for i in range(1, 251):
    username = fake.user_name()
    password = fake.password()
    email = fake.email()
    conn.execute(insert_user_query, (i, username, password, email, datetime.now().isoformat()))

# Insert Products
insert_product_query = "INSERT INTO Products VALUES (?, ?, ?, ?, ?, ?)"
for i in range(1, 101):
    name = fake.word().capitalize()
    description = fake.sentence()
    price = round(random.uniform(5.0, 100.0), 2)
    stock_quantity = random.randint(1, 100)
    conn.execute(insert_product_query, (i, name, description, price, stock_quantity, datetime.now().isoformat()))



In [21]:
from faker import Faker
import random
from datetime import datetime, timedelta

# Initialize
fake = Faker()

# Insert Categories
insert_category_query = "INSERT INTO Categories VALUES (?, ?)"
for i in range(1, 11):
    category_name = fake.word().capitalize()
    conn.execute(insert_category_query, (i, category_name))

# Insert Users
insert_user_query = "INSERT INTO Users VALUES (?, ?, ?, ?, ?)"
for i in range(1, 251):
    username = fake.user_name()
    password = fake.password()
    email = fake.email()
    conn.execute(insert_user_query, (i, username, password, email, datetime.now().isoformat()))

# Insert Products
insert_product_query = "INSERT INTO Products VALUES (?, ?, ?, ?, ?, ?)"
for i in range(1, 101):
    name = fake.word().capitalize()
    description = fake.sentence()
    price = round(random.uniform(5.0, 100.0), 2)
    stock_quantity = random.randint(1, 100)
    conn.execute(insert_product_query, (i, name, description, price, stock_quantity, datetime.now().isoformat()))

# Insert Product Categories
insert_product_category_query = "INSERT INTO Product_Categories VALUES (?, ?)"
for product_id in range(1, 101):
    category_id = random.randint(1, 10)
    conn.execute(insert_product_category_query, (product_id, category_id))

# Insert Orders
insert_order_query = "INSERT INTO Orders VALUES (?, ?, ?, ?)"
for i in range(1, 251):
    user_id = random.randint(1, 250)
    order_date = datetime.now() - timedelta(days=random.randint(1, 30))
    total_amount = round(random.uniform(20.0, 500.0), 2)
    conn.execute(insert_order_query, (i, user_id, order_date.isoformat(), total_amount))

# Insert Order Items
insert_order_item_query = "INSERT INTO Order_Items VALUES (?, ?, ?, ?, ?)"
for order_id in range(1, 251):
    product_id = random.randint(1, 100)
    quantity = random.randint(1, 5)
    price = round(random.uniform(5.0, 100.0), 2)
    conn.execute(insert_order_item_query, (order_id, order_id, product_id, quantity, price))

# Insert Reviews
insert_review_query = "INSERT INTO Reviews VALUES (?, ?, ?, ?, ?, ?)"
for i in range(1, 251):
    product_id = random.randint(1, 100)
    user_id = random.randint(1, 250)
    rating = random.randint(1, 5)
    comment = fake.sentence()
    conn.execute(insert_review_query, (i, product_id, user_id, rating, comment, datetime.now().isoformat()))

In [22]:
conn.commit()

In [60]:
conn.execute("CREATE TABLE GroupMembers (group_member_id INTEGER, group_id INTEGER, user_id INTEGER, joined_at TIMESTAMP)")
insert_group_member_query = "INSERT INTO GroupMembers VALUES (?, ?, ?, ?)"
conn.execute(insert_group_member_query, (1, 1, 1, datetime.now().isoformat()))
for i in range(2, 251):
    group_id = random.randint(1, 50)
    user_id = random.randint(1, 250)
    joined_at = datetime.now().isoformat()
    conn.execute(insert_group_member_query, (i, group_id, user_id, joined_at))


In [24]:
# code = response.python_code.replace('```','').replace('python','')

# # exec(code)
# import duckdb

# If you want to persist the current in-memory tables to "E-commerce store schema.duckdb",
# you need to use the COPY statement or DuckDB's backup/attach functionality.
# Here is how you can save all in-memory tables to a new DuckDB file:

import duckdb

# Assume conn is your current in-memory connection
# conn = duckdb.connect(database=':memory:')

# Persist all in-memory tables to a DuckDB database file called 'ecommerce-dataset.duckdb'

# # Attach the persistent database
# conn.execute("ATTACH 'ecommerce-dataset.duckdb' AS ecommerce_dataset;")

# # Get all in-memory table names
# tables = [row[0] for row in conn.execute("SHOW TABLES").fetchall()]

# # Save each table to the persistent database
# for table in tables:
#     # Create the table in the attached database and copy data
#     conn.execute(f"CREATE TABLE IF NOT EXISTS ecommerce_dataset.{table} AS SELECT * FROM {table};")

# Optionally, commit changes to ensure all data is written
conn.commit()

# Get all table names in memory
# tables = conn.execute("SHOW TABLES").fetchall()
# for (table_name,) in tables:
#     count = conn.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
#     print(f"{table_name}: {count} rows")

# Now, you can connect directly to the persistent database in the future:
# conn = duckdb.connect(database='E-commerce store schema.duckdb')

# Show tables and row counts in the persistent database
# tables = conn.execute("SHOW TABLES IN ecommerce_db").fetchall()
# print("Tables and row counts in 'E-commerce store schema.duckdb':")
# for (table_name,) in tables:
#     count = conn.execute(f"SELECT COUNT(*) FROM ecommerce_db.{table_name}").fetchone()[0]
#     print(f"{table_name}: {count} rows")



In [29]:
import duckdb
# conn.close()
# Connect to the persistent DuckDB database
conn = duckdb.connect(database='ecommerce-dataset.duckdb')

# Get all table names
tables = conn.execute("SHOW TABLES").fetchall()

# Print table names and row counts
print("Tables and row counts in 'ecommerce-dataset.duckdb':")
for (table_name,) in tables:
    count = conn.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    print(f"{table_name}: {count} rows")
conn.close()

Tables and row counts in 'ecommerce-dataset.duckdb':
Categories: 10 rows
Order_Items: 250 rows
Orders: 250 rows
Product_Categories: 100 rows
Products: 100 rows
Reviews: 250 rows
Users: 250 rows


In [ ]:
import dspy

class sql_generator(dspy.Signature):
    """
    Given a natural language question and a database schema, generate a syntactically correct SQL query
    compatible with DuckDB that accurately answers the question. Use only the tables and columns explicitly 
    present in the provided schema. Ensure proper SQL syntax, including JOINs, WHERE clauses, GROUP BY, 
    ORDER BY, and aggregation functions when necessary.

    Follow these constraints:
    - Do not assume relationships between tables unless they are clearly defined by column names (e.g., foreign keys).
    - Use fully qualified column names when querying from multiple tables (e.g., table_name.column_name).
    - Use DuckDB-compatible syntax (similar to PostgreSQL), avoiding unsupported functions.
    - Apply LIMIT clauses for large or unbounded result sets if appropriate.
    - Do not return explanatory text — only the SQL query.

    Example:
    Question:
    "List the top 5 products (by name) that generated the highest total revenue in 2024."

    Schema:
    Table: products (
        id INTEGER,
        name TEXT,
        category TEXT,
        price FLOAT
    )

    Table: orders (
        id INTEGER,
        order_date DATE,
        customer_id INTEGER
    )

    Table: order_items (
        id INTEGER,
        order_id INTEGER,
        product_id INTEGER,
        quantity INTEGER
    )

    Expected SQL:
    SELECT products.name, SUM(products.price * order_items.quantity) AS total_revenue
    FROM order_items
    JOIN products ON order_items.product_id = products.id
    JOIN orders ON order_items.order_id = orders.id
    WHERE EXTRACT(YEAR FROM orders.order_date) = 2024
    GROUP BY products.name
    ORDER BY total_revenue DESC
    LIMIT 5;
    """

    question = dspy.InputField(desc="The user's natural language question about the database.")
    sql_schema = dspy.InputField(desc="The schema of the database (tables, columns, types, etc).")
    sql = dspy.OutputField(desc="A syntactically correct SQL query (DuckDB-compatible) that answers the question.")


class sql_corrector(dspy.Signature):
    """
    Given a faulty SQL query, the original user question, database schema, and the associated error message,
    return a corrected SQL query that is syntactically valid in DuckDB and semantically answers the original question.

    Constraints:
    - Use only the tables and columns provided in the schema.
    - Fix only what is necessary based on the error message.
    - Follow DuckDB-compatible SQL syntax (similar to PostgreSQL).
    - Do not assume table relationships unless foreign keys or naming conventions clearly imply them.
    - Always ensure joins are explicitly written and column references are unambiguous.
    - Do not return any explanation — only the corrected SQL query.

    Example:
    Question:
    "Show each customer's name along with the total amount they spent on orders in 2023."

    Schema:
    Table: customers (
        id INTEGER,
        name TEXT,
        email TEXT
    )

    Table: orders (
        id INTEGER,
        customer_id INTEGER,
        order_date DATE
    )

    Table: order_items (
        id INTEGER,
        order_id INTEGER,
        product_id INTEGER,
        quantity INTEGER,
        unit_price FLOAT
    )

    Faulty SQL:
    SELECT name, SUM(quantity * unit_price) AS total_spent
    FROM customers
    JOIN orders ON customers.id = orders.customer_id
    JOIN order_items ON orders.id = order_id
    WHERE EXTRACT(YEAR FROM order_date) = 2023
    GROUP BY name;

    Error (DuckDB):
    Binder Error: Column "order_id" not found in scope

    Corrected SQL:
    SELECT customers.name, SUM(order_items.quantity * order_items.unit_price) AS total_spent
    FROM customers
    JOIN orders ON customers.id = orders.customer_id
    JOIN order_items ON orders.id = order_items.order_id
    WHERE EXTRACT(YEAR FROM orders.order_date) = 2023
    GROUP BY customers.name;
    """

    question = dspy.InputField(desc="The user's original natural language question.")
    sql_schema = dspy.InputField(desc="The schema of the database (tables, columns, types, etc).")
    faulty_sql = dspy.InputField(desc="The SQL query that failed to execute.")
    error = dspy.InputField(desc="The error message returned when executing the faulty SQL.")
    corrected_sql = dspy.OutputField(desc="A corrected SQL query that should execute successfully and answer the question.")



In [ ]:


class text2sqlagent(dspy.Module):
    def __init__(self):
        
        self.difficulty_retries ={'basic':1 , 'intermediate':2, 'advanced':4}
        self.sql_genator_agent = dspy.asyncify(dspy.Predict(sql_generator))
        self.sql_corrector = dspy.asyncify(dspy.Predict(sql_corrector))
        self.basic_lm = dspy.LM(model='openai/gpt-4o-mini', api_key=os.getenviron("OPENAI_API_KEY"),max_tokens=5000)
        self.intermediate_lm = dspy.LM('anthropic/claude-3.5-sonnet',api_key=os.getenviron("ANTHROPIC_API_KEY") max_tokens =5000)
        self.advanced_lm = dspy.LM('gemini/gemini-2.5-pro', api_key=os.getenviron("GEMINI_API_KEY"), max_token=7000)
        self.llm_dictionary = {'basic':self.basic_lm, 'intermediate':self.intermediate_lm, 'advanced':self.advanced_lmm}


    async def aforward(self, difficulty, question, schema, conn):
        lm = self.llm_dictionary[difficulty.lower()]
        loop = self.difficulty_retries[difficulty.lower()]
        previous_sql = ''
        error = ''
        is_executable = False
        result = ''


        with dspy.context(lm=lm):
            for i in range(loop):
                if i == 0:
                    response = await self.sql_genator_agent(question=question, sql_schema=schema)
                    sql = response.sql.replace('```', '').replace('sql', '')
                else:
                    response = await self.sql_corrector(
                        question=question,
                        sql_schema=schema,
                        faulty_sql=previous_sql,
                        error=error
                    )
                    sql = response.corrected_sql.replace('```', '').replace('sql', '')
                try:
                    result = conn.execute(sql).fetchall()
                    result = str(result)
                    is_executable = True
                    break
                except Exception as e:
                    is_executable = False
                    error = str(e)
                    previous_sql = sql
                    pass
        return {'sql':sql, 'result':result, 'is_executable':is_executable}
                

            
            

                    

            






        
        




In [ ]:
class basic_question_gen(dspy.Signature):
    """
    You are part of an AI-powered SQL training system designed to help beginners learn SQL through guided practice.

    Given:
    - A DuckDB database schema (`db_schema`) that includes tables and columns.
    - An optional topic (`topic`) such as SELECT, WHERE, JOIN, GROUP BY, etc.

    Your task:
    - Generate 1 beginner-level SQL question based on the provided schema and topic.
    - If the topic is 'All', select a fundamental concept like:
        - Selecting columns
        - Filtering rows
        - Sorting data
        - Using COUNT or SUM
        - Applying LIMIT

    Output:
    - A clear and simple question related to the schema.
    - A correct SQL solution for that question.
    """
    db_schema = dspy.InputField(desc="The schema of the DuckDB database")
    topic = dspy.InputField(desc="The SQL topic user wants to learn", default="All")
    question = dspy.OutputField(desc="A single basic-level SQL question")
    solution_sql = dspy.OutputField(desc="Correct SQL query that solves the question")


class intermediate_question_gen(dspy.Signature):
    """
    You are part of an AI-powered SQL training system designed to help users advance their SQL skills through practical exercises.

    Given:
    - A DuckDB database schema (`db_schema`) that includes tables and columns.
    - An optional topic (`topic`) such as JOINs, GROUP BY, subqueries, etc.

    Your task:
    - Generate 1 intermediate-level SQL question that applies concepts like:
        - JOINs across tables
        - GROUP BY with aggregate functions
        - Subqueries in SELECT or WHERE
        - Filtering using IN, BETWEEN, LIKE
        - HAVING clause

    Output:
    - A clear intermediate-level question that challenges understanding.
    - A correct SQL solution for that question.
    """
    db_schema = dspy.InputField(desc="The schema of the DuckDB database")
    topic = dspy.InputField(desc="The SQL topic user wants to learn", default="All")
    question = dspy.OutputField(desc="A single intermediate-level SQL question")
    solution_sql = dspy.OutputField(desc="Correct SQL query that solves the question")


class hard_question_gen(dspy.Signature):
    """
    You are part of an AI-powered SQL training system designed to help users master advanced SQL through challenging problems.

    Given:
    - A DuckDB database schema (`db_schema`) that includes tables and columns.
    - An optional topic (`topic`) such as advanced JOINs, window functions, CTEs, etc.

    Your task:
    - Generate 1 advanced SQL question involving:
        - CTEs (WITH clause)
        - Window functions (RANK, ROW_NUMBER, etc.)
        - Correlated subqueries
        - Multi-level aggregation
        - INTERSECT, EXCEPT

    Output:
    - A realistic and challenging SQL question.
    - A valid SQL solution that solves the problem.
    """
    db_schema = dspy.InputField(desc="The schema of the DuckDB database")
    topic = dspy.InputField(desc="The SQL topic user wants to learn", default="All")
    question = dspy.OutputField(desc="A single hard-level SQL question")
    solution_sql = dspy.OutputField(desc="Correct SQL query that solves the question")


class explanation_gen(dspy.Signature):
    """
    You are part of an AI-powered SQL training system that helps users understand and learn from their mistakes.

    Given:
    - `error_generated`: The error message returned by the SQL engine (DuckDB) after executing a query.
    - `faulty_sql`: The original SQL query written by the user that caused the error.

    Your task:
    - Analyze the error message and the SQL query.
    - Generate a clear, beginner-friendly explanation of what went wrong in the SQL query.
    - Avoid technical jargon where possible.
    - Focus on helping the user understand the mistake so they can learn and fix it.

    Format:
    - Output a single paragraph in simple English.
    - Use analogies or examples if it helps clarify the issue.
    """
    error_generated = dspy.InputField(desc="The error generated by the system")
    faulty_sql = dspy.InputField(desc="The SQL user entered into the system")
    explanation = dspy.OutputField(desc="A simple language explanation for the faulty SQL")

In [ ]:
# !pip uninstall docx



In [ ]:

# !pip install pypandoc

# Get the list of all tables
tables = [row[0] for row in conn.execute("SHOW TABLES").fetchall()]

table_heads = {}
for table in tables:
    df_head = conn.execute(f"SELECT * FROM {table}").df().head()
    table_heads[table] = df_head

# Now table_heads is a dictionary mapping table names to their head DataFrames